# SKU forecasting quickstart

You can [download this notebook](https://docs.generalforecast.com/notebooks/sku_forecasting_quickstart.ipynb) and run it in Jupyter, or
[open it in Google Colab](https://colab.research.google.com/github/generalforecast/gfi/blob/main/examples/sku_forecasting_quickstart.ipynb). Everything here has been run locally and takes about two
minutes on a MacBook Pro's CPU once GF is installed, including downloading the data.

In this notebook, we'll forecast the next four weeks of demand for the 500 best-selling products
at one Walmart store (CA_1), with a point forecast and the 10-90% quantile range. Each product
(SKU) has 150 weeks of sales history, and we give each SKU its own small table, so each forecast
reads that product's own history. There's no training and no tuning -- GF's model reads the
table when it forecasts, and new weeks of sales go straight into it.

## Install

To install GF, follow the steps in our [quickstart](https://docs.generalforecast.com/get-started/quickstart/).

In Jupyter or Colab, the cell below installs it from GitHub, along with pandas and matplotlib.

In [ ]:
# Installs generalforecast from GitHub, which needs read access to the generalforecast repository: a GITHUB_TOKEN
# secret in Colab (the key icon on the left), and a GITHUB_TOKEN environment variable or your
# git credentials elsewhere. Building generalforecast needs Rust, which this cell installs in Colab.
import importlib.util
import os
import shutil
import subprocess
import sys

# Packages go into the Python this notebook's kernel runs, which generalforecast needs to be 3.12 or newer.
print(f"Python {sys.version.split()[0]} at {sys.executable}")
if sys.version_info < (3, 12):  # noqa: UP036 (the kernel, not this repository, picks the Python)
    raise SystemExit(
        "generalforecast needs Python 3.12 or newer. Pick a kernel with a newer Python."
    )


def install(*packages):
    """Installs into this notebook's environment, with pip, or uv where there is no pip."""
    if importlib.util.find_spec("pip") is not None:
        command = [sys.executable, "-m", "pip", "install", "-q"]
    elif shutil.which("uv") is not None:
        command = ["uv", "pip", "install", "-q", "--python", sys.executable]
    else:
        raise SystemExit(
            "this environment has neither pip nor uv to install packages with"
        )
    subprocess.run(
        [*command, *packages],
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if importlib.util.find_spec("generalforecast") is None:
    colab = "google.colab" in sys.modules
    if colab:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        os.environ["PATH"] += ":/root/.cargo/bin"
        subprocess.run(
            "curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q",
            shell=True,
            check=True,
        )
    else:
        token = os.environ.get("GITHUB_TOKEN")
    login = f"{token}@" if token else ""
    install(
        f"generalforecast[sklearn] @ git+https://{login}github.com/generalforecast/gfi@main"
    )
    if colab:
        print(
            "generalforecast installed. Restarting to load it. Run this cell again after."
        )
        os.kill(os.getpid(), 9)
missing = [
    name for name in ("pandas", "matplotlib") if importlib.util.find_spec(name) is None
]
if missing:
    install(*missing)

## 1. Load the data

We use the M5 forecasting competition's Walmart data, from Nixtla's public copy (50 MB). It comes
as three tables, and the sales of the four weeks after them, which the competition released later
and which we keep aside to check our forecast against at the end.

In [ ]:
import os
import urllib.request
import zipfile

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import generalforecast

if not os.path.exists("m5.zip"):
    urllib.request.urlretrieve(
        "https://github.com/Nixtla/m5-forecasts/raw/main/datasets/m5.zip", "m5.zip"
    )
with zipfile.ZipFile("m5.zip") as archive:
    sales = pd.read_csv(archive.open("sales_train_evaluation.csv"))
    calendar = pd.read_csv(archive.open("calendar.csv"), parse_dates=["date"])
    prices = pd.read_csv(archive.open("sell_prices.csv"))
    released = pd.read_csv(archive.open("sales_test_evaluation.csv"))
# The calendar's rows are the days d_1, d_2, ... in order.
calendar.insert(0, "d", [f"d_{day}" for day in range(1, len(calendar) + 1)])

`sales` has one row per product and store, and one column per day of units sold, `d_1` to
`d_1941`. `released` has the same rows for the 28 days after, `d_1942` to `d_1969`.

In [ ]:
sales.iloc[:5, :9]

`calendar` maps each day `d_1`, `d_2`, ... to its date and Walmart's week number, `wm_yr_wk`
(`11101` is week 1 of Walmart's 2011, and Walmart weeks run Saturday to Friday), with any event
that day and whether SNAP (food stamp) payments went out in each state.

In [ ]:
calendar.head()

`prices` has each product's price at each store in each Walmart week.

In [ ]:
prices.head()

## 2. Setting the forecast parameters

These name the columns of the table we build in step 3 and set how far to forecast. `HORIZON` is
how many weeks to forecast, and `HISTORY` how many weeks each SKU's table holds.

In [ ]:
SERIES = "item_id"  # the column that names a SKU
DATE = "date"  # the week, as a date
TARGET = "units"  # what to forecast; empty in the weeks to forecast
KNOWN = ["price", "events", "snap_days"]  # columns known ahead of time
SEASON = 52  # periods in a year: 52 for weekly data, 12 for monthly
HORIZON = 4  # periods to forecast
HISTORY = 150  # periods of history per SKU

## 3. Preprocessing

GF reads a table with one row per observation, and we forecast weekly demand, so we need one
row per SKU and week. First, we keep store CA_1 and its 500 best-selling products, add the
released days, and turn the day columns into rows.

In [ ]:
store = sales[sales["store_id"] == "CA_1"]
top = store.set_index("item_id").filter(like="d_").sum(axis=1).nlargest(500).index
store = store.merge(
    released.drop(columns=["dept_id", "cat_id", "state_id"]), on=["item_id", "store_id"]
)
days = [column for column in store.columns if column.startswith("d_")]
daily = store[store["item_id"].isin(top)].melt(
    id_vars="item_id", value_vars=days, var_name="d", value_name="units"
)
daily.head()

Next, we give every day its Walmart week, and add each SKU's days up into weeks, because weekly
totals are what a planner orders against and they are less noisy than days. Each week gets its
first day as its date, and counts how many of its days had an event or SNAP payments, because
both move demand and are known ahead of time. Weeks the data covers only partly are dropped, as
their totals would look like a drop in sales.

In [ ]:
calendar["event"] = calendar["event_name_1"].notna()
weeks = calendar.groupby("wm_yr_wk").agg(
    date=("date", "min"),
    days=("date", "size"),
    events=("event", "sum"),
    snap_days=("snap_CA", "sum"),
)
weeks = weeks[weeks["days"] == 7].drop(columns="days")

daily = daily.merge(calendar[["d", "wm_yr_wk"]], on="d")
weekly = daily.groupby(["item_id", "wm_yr_wk"])["units"].agg(["sum", "size"])
weekly = weekly[weekly["size"] == 7]["sum"].rename("units").reset_index()
weekly.head()

Then we put every SKU next to every week, which gives the weeks we forecast a row each, and add
each week's price, which Walmart sets ahead of time and which moves demand. The released weeks are
the ones we forecast, so their sales move to an `actual` column, which we compare against in step
7. We no longer need the raw tables, so we free their memory.

This table is the input for our forecasts using GF. GF reads `date` itself, as days since 1970,
weekday, day of month and month.

In [ ]:
grid = pd.MultiIndex.from_product([top, weeks.index], names=["item_id", "wm_yr_wk"])
data = grid.to_frame(index=False).merge(weeks, left_on="wm_yr_wk", right_index=True)
data = data.merge(weekly, how="left")
data = data.merge(
    prices[prices["store_id"] == "CA_1"][["item_id", "wm_yr_wk", "sell_price"]],
    how="left",
).rename(columns={"sell_price": "price"})
first_released = calendar.loc[calendar["d"] == "d_1942", "wm_yr_wk"].iloc[0]
released_weeks = data["wm_yr_wk"] >= first_released
data["actual"] = data["units"].where(released_weeks)
data.loc[released_weeks, "units"] = np.nan
data = data.drop(columns="wm_yr_wk")
del sales, released, prices, store, daily, weekly
FEATURES = [DATE, *KNOWN]
data[data[SERIES] == top[0]].tail(8)

Last, we split the weeks. Each SKU's last `HORIZON` weeks of sales are held out to check the
forecasts against in step 5, the `HISTORY` weeks before them go into its table, and the `HORIZON`
weeks after the sales are the ones we forecast in step 7.

In [ ]:
history = data[data[TARGET].notna()]
last = history[DATE].max()
cutoff = last - pd.Timedelta(weeks=HORIZON - 1)
held_out = history[history[DATE] >= cutoff]
known = history[
    (history[DATE] < cutoff) & (history[DATE] >= cutoff - pd.Timedelta(weeks=HISTORY))
]
future = data[(data[DATE] > last) & (data[DATE] <= last + pd.Timedelta(weeks=HORIZON))]
print(
    f"{known[SERIES].nunique()} SKUs: {len(known):,} weeks of history, {len(held_out):,} held out, {len(future):,} to forecast"
)

## 4. One table per SKU

Every SKU gets a table of its history. All 500 tables share one model, and one `predict` call
forecasts them together.

In [ ]:
client = generalforecast.local()
tables = {}
for sku, rows in known.groupby(SERIES):
    tables[sku] = client.create_table(
        sku, model="general-forecast/tabicl-v2-regressor", columns=rows[FEATURES]
    )
    tables[sku].insert(rows[DATE].astype(str), rows[FEATURES], rows[TARGET])


def forecast(rows):
    """Each row's 10th, 50th and 90th percentiles, from its SKU's table, at least 0 units."""
    quantiles = client.predict(
        {sku: group[FEATURES] for sku, group in rows.groupby(SERIES)}
    )
    # A regressor returns 999 quantiles, the k-th at level (k + 1) / 1000.
    picked = [
        pd.DataFrame(
            quantiles[sku][:, [99, 499, 899]],
            index=group.index,
            columns=["low", "forecast", "high"],
        )
        for sku, group in rows.groupby(SERIES)
    ]
    return pd.concat(picked).loc[rows.index].clip(lower=0)


print(f"{len(tables)} tables of {len(known) / len(tables):.0f} weeks each")

## 5. How accurate is it?

We forecast the held-out weeks and compare the mean absolute error with three simple forecasts
planners often use: each SKU's last week, its average over the last 13 weeks, and the same week
last year. We also check how often the actual sales fall within the 10-90% range.

We compared GF with XGBoost and LightGBM on the same SKUs, weeks and features in
[`m5_sku_forecasting.ipynb`](m5_sku_forecasting.ipynb). GF's error was lower than both, 10.81
units a week against 11.19 for XGBoost and 11.46 for LightGBM, with no training.

In [ ]:
METHODS = {
    "GF": "forecast",
    "last week": "last week",
    "13-week average": "13-week average",
    "same week last year": "same week last year",
}


def score(rows, before, actual):
    """Adds the baselines from the weeks `before` to `rows`, and prints each method's error."""
    recent = before.sort_values(DATE).groupby(SERIES)[TARGET]
    rows["last week"] = rows[SERIES].map(recent.last())
    rows["13-week average"] = rows[SERIES].map(
        recent.apply(lambda units: units.tail(13).mean())
    )
    year_ago = pd.MultiIndex.from_arrays(
        [rows[SERIES], rows[DATE] - pd.Timedelta(weeks=SEASON)]
    )
    rows["same week last year"] = (
        data.set_index([SERIES, DATE])[TARGET].reindex(year_ago).to_numpy()
    )
    errors = {
        name: (rows[actual] - rows[column]).abs().mean()
        for name, column in METHODS.items()
    }
    table = pd.DataFrame({"mean absolute error": errors}).round(2)
    table["GF's error is lower by"] = [""] + [
        f"{1 - errors['GF'] / error:.0%}" for error in list(errors.values())[1:]
    ]
    inside = rows[actual].between(rows["low"], rows["high"]).mean()
    print(
        f"{table.to_string()}\n\n{inside:.0%} of actual weeks fall within GF's 10-90% range"
    )
    return rows


checked = score(held_out.join(forecast(held_out)), known, TARGET)

The next cell sets the charts' style and defines `compare`, which draws a SKU's recent and actual
sales in black and each method's forecast in its own color.

In [ ]:
INK, GRID, PAPER = "#191a1c", "#dcdcd8", "#f3f3f1"
STYLE = {  # a color and marker for each method
    "GF": ("#2a6fdb", "o"),
    "last week": ("#e8833a", "s"),
    "13-week average": ("#2ca58d", "D"),
    "same week last year": ("#b04fbf", "^"),
}
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Neue Montreal", "Helvetica Neue", "Arial", "DejaVu Sans"],
        "font.size": 12,
        "figure.facecolor": PAPER,
        "axes.facecolor": PAPER,
        "savefig.facecolor": PAPER,
        "axes.edgecolor": "#9aa0a6",
        "text.color": INK,
        "xtick.color": "#5f6368",
        "ytick.color": "#5f6368",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": GRID,
        "grid.linewidth": 0.8,
    }
)


def compare(sku, rows, actual, title):
    """The SKU's 12 weeks before `rows` and its actual sales, with every method's forecast of `rows`."""
    one = rows[rows[SERIES] == sku]
    before = data[(data[SERIES] == sku) & (data[DATE] < one[DATE].min())].tail(12)
    fig, ax = plt.subplots(figsize=(13, 6.5))
    ax.axvspan(
        one[DATE].min() - pd.Timedelta(days=3),
        one[DATE].max() + pd.Timedelta(days=3),
        color=GRID,
        alpha=0.6,
        linewidth=0,
    )
    color = STYLE["GF"][0]
    ax.fill_between(
        one[DATE],
        one["low"],
        one["high"],
        color=color,
        alpha=0.12,
        linewidth=0,
        label="GF's 10-90% range",
    )
    ax.plot(
        pd.concat([before[DATE], one[DATE]]),
        pd.concat([before[TARGET], one[actual]]),
        color=INK,
        linewidth=2.5,
        marker="o",
        markersize=6,
        label="actual sales",
        zorder=3,
    )
    for name, column in METHODS.items():
        color, marker = STYLE[name]
        error = (one[actual] - one[column]).abs().mean()
        ax.plot(
            one[DATE],
            one[column],
            color=color,
            linewidth=2,
            marker=marker,
            markersize=7,
            label=f"{name}, off by {error:.0f} a week",
        )
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    ax.set_title(f"{sku}: {title}", loc="left", fontsize=14)
    ax.set_ylabel("units sold a week")
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(ax.xaxis.get_major_locator())
    )
    fig.tight_layout()
    plt.show()

Here is the best-selling SKU among those whose held-out weeks GF forecast more closely than
every baseline, with its 12 weeks before them and the four held-out weeks (shaded).

In [ ]:
errors = checked[list(METHODS.values())].sub(checked[TARGET], axis=0).abs()
best = errors.groupby(checked[SERIES]).mean().idxmin(axis=1)
example = next(sku for sku in top if best[sku] == "forecast")
compare(example, checked, TARGET, "forecasts of the four held-out weeks")

## 6. New sales arrive

The held-out weeks have happened, so we insert them into each SKU's table. Nothing is retrained,
and the next forecast reads them.

In [ ]:
for sku, rows in held_out.groupby(SERIES):
    tables[sku].insert(rows[DATE].astype(str), rows[FEATURES], rows[TARGET])
print(
    f"{sum(len(table) for table in tables.values()):,} rows across {len(tables)} tables"
)

## 7. Forecast the next four weeks

These are the weeks after the sales data ends, with their known prices and events, for every
SKU. The forecasts and their ranges go to `forecast.csv`. M5 released these weeks' sales after
the competition, so we can score this forecast against what actually sold.

In [ ]:
ahead = future.join(forecast(future))
ahead[[SERIES, DATE, "low", "forecast", "high"]].to_csv("forecast.csv", index=False)
ahead = score(ahead, history, "actual")

In [ ]:
compare(
    example,
    ahead,
    "actual",
    "forecasts of the next four weeks, against the sales M5 released",
)